In [2]:
import sys
print('Kernel executable:', sys.executable)
print('Python version:', sys.version.split()[0])

%pip install -r ../requirements.txt
%pip install scikit-fuzzy networkx

Kernel executable: c:\repos\adni-project\.venv\Scripts\python.exe
Python version: 3.11.6
Note: you may need to restart the kernel to use updated packages.
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/2.1 MB ? eta -:--:--
   ---------- ----------------------------- 0.5/2.1 MB 1.3 MB/s eta 0:00:02
   --------------- ------------------------ 0.8/2.1 MB 1.3 MB/s eta 0:00:01
   -------------------- ------------------- 1.0/2.1 MB 1.3 MB/s eta 0:00:01
   ------------------------- -------------- 1.3/2.1 MB 1.3 MB/s eta 0:00:01
   ------------------------------ --------- 1.6/2.1 MB 1.3 MB/s eta 0:00:01
   ----------------------------------- ---- 1.8/2.1 MB 1.3 MB/s eta 0:00:01
   ---------------------------------------- 2.1/2.1 MB 1.3 MB/s  0:00:01
Note: you may need to restart the kernel to use updated packages.


## Runtime And Dependency Setup

Use Python 3.11 kernel (`.venv`) for this notebook.

If imports fail, run the next cell once, then restart the kernel and run all cells top to bottom.

# Hybrid Fuzzy Refinement for AD Progression

This notebook adds an interpretable fuzzy layer on top of processed ADNI longitudinal features.

Inputs used:
- `MMSE` (scaled 0-1 in preprocessing output)
- `CDRSB` (scaled 0-1)
- `HIPP_TOTAL` (scaled 0-1)

Output:
- `RISK_SCORE` in [0, 1], converted to stage predictions (`CN`, `MCI`, `AD`)

In [3]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

import skfuzzy as fuzz
from skfuzzy import control as ctrl

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [4]:
OUT_DIR = Path('..') / 'outputs'
train_path = OUT_DIR / 'train_longitudinal_clean.csv'
test_path = OUT_DIR / 'test_longitudinal_clean.csv'

required_paths = [
    train_path,
    test_path,
    OUT_DIR / 'X_train.npy',
    OUT_DIR / 'X_test.npy',
    OUT_DIR / 'y_train_stage.npy',
    OUT_DIR / 'y_test_stage.npy',
]

missing_paths = [str(p) for p in required_paths if not p.exists()]
if missing_paths:
    raise FileNotFoundError(
        'Missing required files. Run notebooks/exploration.ipynb first. Missing:\n' + '\n'.join(missing_paths)
    )

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

required_cols = ['PTID', 'VISCODE', 'VISMONTH', 'MMSE', 'CDRSB', 'HIPP_TOTAL']
missing_cols = [c for c in required_cols if c not in test_df.columns]
if missing_cols:
    raise ValueError(f'Test dataframe is missing required columns: {missing_cols}')

X_train = np.load(OUT_DIR / 'X_train.npy')
X_test = np.load(OUT_DIR / 'X_test.npy')
y_train_stage = np.load(OUT_DIR / 'y_train_stage.npy').astype(int)
y_test_stage = np.load(OUT_DIR / 'y_test_stage.npy').astype(int)

print('Train shape:', train_df.shape)
print('Test shape :', test_df.shape)
print('X_train shape:', X_train.shape)
print('X_test shape :', X_test.shape)
print('y_train_stage shape:', y_train_stage.shape)
print('y_test_stage shape :', y_test_stage.shape)
display(test_df.head())

Train shape: (6247, 12)
Test shape : (1473, 12)
X_train shape: (2140, 3, 8)
X_test shape : (496, 3, 8)
y_train_stage shape: (2140,)
y_test_stage shape : (496,)


,PTID,VISCODE,VISMONTH,MMSE,ADAS13,CDRSB,HIPP_TOTAL,AGE,PTGENDER,PTEDUCAT,APOE4,STAGE_CODE
0,002_S_2132,sc,-1.0,0.903226,0.130288,0.144444,0.205473,0.362466,0.0,1.0000,0.2,1.0
1,002_S_4473,init,-1.0,0.967742,0.087106,0.055556,0.168806,0.602100,0.0,0.7500,0.5,1.0
2,002_S_6066,sc,-1.0,1.000000,0.192652,0.000000,0.164412,0.443496,0.0,1.0000,0.5,0.0
3,002_S_6066,bl,0.0,0.922581,0.073345,0.111111,0.179737,0.443496,0.0,1.0000,0.5,0.0
4,002_S_6070,sc,-1.0,0.967742,0.079840,0.000000,0.198050,0.398171,0.0,0.9375,0.2,0.0


## Build fuzzy inference system

Membership design assumptions:
- Lower MMSE means higher risk
- Higher CDRSB means higher risk
- Lower hippocampal volume means higher risk

## Fuzzy math used in this notebook

> This section documents the exact fuzzy relations implemented in the next code cell.

### 1) Triangular membership function
For each variable (`mmse`, `cdrsb`, `hipp`, `risk`) and each linguistic set (`low`, `mid`, `high`), we use a triangular MF:

$$\mu_{\mathrm{tri}}(x; a,b,c)=\max\left(\min\left(\frac{x-a}{b-a},\frac{c-x}{c-b}\right),0\right)$$

In code this is implemented with `fuzz.trimf(u, [a, b, c])` where `u = np.linspace(0,1,101)`.

### 2) Fuzzy operators and inference
- `AND` in rules uses t-norm (minimum).
- `OR` in rules uses s-norm (maximum).
- Rule implication clips consequent MF by antecedent firing strength.
- Aggregation combines rule outputs by maximum.
- Defuzzification uses centroid to return a crisp `risk` score in `[0,1]`.

### 3) Crisp stage mapping from risk
The direct fuzzy-only stage mapping uses:
- `CN (0)` if `risk < 0.33`
- `MCI (1)` if `0.33 <= risk < 0.66`
- `AD (2)` if `risk >= 0.66`

In [5]:
u = np.linspace(0, 1, 101)

mmse = ctrl.Antecedent(u, 'mmse')
cdrsb = ctrl.Antecedent(u, 'cdrsb')
hipp = ctrl.Antecedent(u, 'hipp')
risk = ctrl.Consequent(u, 'risk')

mmse['low'] = fuzz.trimf(u, [0.0, 0.0, 0.45])
mmse['mid'] = fuzz.trimf(u, [0.25, 0.5, 0.75])
mmse['high'] = fuzz.trimf(u, [0.55, 1.0, 1.0])

cdrsb['low'] = fuzz.trimf(u, [0.0, 0.0, 0.35])
cdrsb['mid'] = fuzz.trimf(u, [0.2, 0.5, 0.8])
cdrsb['high'] = fuzz.trimf(u, [0.6, 1.0, 1.0])

hipp['low'] = fuzz.trimf(u, [0.0, 0.0, 0.4])
hipp['mid'] = fuzz.trimf(u, [0.25, 0.5, 0.75])
hipp['high'] = fuzz.trimf(u, [0.6, 1.0, 1.0])

risk['low'] = fuzz.trimf(u, [0.0, 0.0, 0.35])
risk['mid'] = fuzz.trimf(u, [0.2, 0.5, 0.8])
risk['high'] = fuzz.trimf(u, [0.65, 1.0, 1.0])

rules = [
    ctrl.Rule(mmse['high'] & cdrsb['low'] & hipp['high'], risk['low']),
    ctrl.Rule(mmse['mid'] | cdrsb['mid'] | hipp['mid'], risk['mid']),
    ctrl.Rule(mmse['low'] | cdrsb['high'] | hipp['low'], risk['high']),
    ctrl.Rule(mmse['low'] & cdrsb['high'], risk['high']),
    ctrl.Rule(mmse['high'] & cdrsb['high'], risk['mid']),
]

risk_system = ctrl.ControlSystem(rules)
risk_sim = ctrl.ControlSystemSimulation(risk_system)

In [ ]:
membership_params = pd.DataFrame(
    [
        ['mmse', 'low', 0.00, 0.00, 0.45],
        ['mmse', 'mid', 0.25, 0.50, 0.75],
        ['mmse', 'high', 0.55, 1.00, 1.00],
        ['cdrsb', 'low', 0.00, 0.00, 0.35],
        ['cdrsb', 'mid', 0.20, 0.50, 0.80],
        ['cdrsb', 'high', 0.60, 1.00, 1.00],
        ['hipp', 'low', 0.00, 0.00, 0.40],
        ['hipp', 'mid', 0.25, 0.50, 0.75],
        ['hipp', 'high', 0.60, 1.00, 1.00],
        ['risk', 'low', 0.00, 0.00, 0.35],
        ['risk', 'mid', 0.20, 0.50, 0.80],
        ['risk', 'high', 0.65, 1.00, 1.00],
    ],
    columns=['variable', 'set', 'a', 'b', 'c']
 )

rule_text = [
    'R1: IF mmse is high AND cdrsb is low AND hipp is high THEN risk is low',
    'R2: IF mmse is mid OR cdrsb is mid OR hipp is mid THEN risk is mid',
    'R3: IF mmse is low OR cdrsb is high OR hipp is low THEN risk is high',
    'R4: IF mmse is low AND cdrsb is high THEN risk is high',
    'R5: IF mmse is high AND cdrsb is high THEN risk is mid',
]

print('Fuzzy membership parameters (trimf control points):')
display(membership_params)
print('Fuzzy rule base:')
for r in rule_text:
    print('-', r)

In [14]:
def fuzzy_risk_score(mmse_val, cdrsb_val, hipp_val):
    risk_sim.input['mmse'] = float(np.clip(mmse_val, 0.0, 1.0))
    risk_sim.input['cdrsb'] = float(np.clip(cdrsb_val, 0.0, 1.0))
    risk_sim.input['hipp'] = float(np.clip(hipp_val, 0.0, 1.0))
    risk_sim.compute()
    return float(risk_sim.output['risk'])

def risk_to_stage(r):
    if r < 0.33:
        return 0
    if r < 0.66:
        return 1
    return 2

test_df['RISK_SCORE'] = test_df.apply(
    lambda row: fuzzy_risk_score(row['MMSE'], row['CDRSB'], row['HIPP_TOTAL']),
    axis=1
)
test_df['FUZZY_STAGE_PRED'] = test_df['RISK_SCORE'].map(risk_to_stage)

SEQUENCE_LENGTH = int(X_train.shape[1])
feature_cols = ['MMSE', 'ADAS13', 'CDRSB', 'HIPP_TOTAL', 'AGE', 'PTGENDER', 'PTEDUCAT', 'APOE4']

def aligned_target_rows(df, sequence_length=3):
    rows = []
    for _, group in df.groupby('PTID'):
        group = group.sort_values(['VISMONTH', 'VISCODE'])
        vals = group[feature_cols].values
        if len(vals) >= sequence_length + 1:
            for i in range(len(vals) - sequence_length):
                rows.append(group.iloc[i + sequence_length])
    if not rows:
        return pd.DataFrame(columns=df.columns)
    return pd.DataFrame(rows).reset_index(drop=True)

target_rows_train = aligned_target_rows(train_df, sequence_length=SEQUENCE_LENGTH)
target_rows_test = aligned_target_rows(test_df, sequence_length=SEQUENCE_LENGTH)

if len(target_rows_train) != len(y_train_stage):
    raise ValueError(
        f'Train target row count ({len(target_rows_train)}) does not match y_train_stage length ({len(y_train_stage)}).'
    )
if len(target_rows_test) != len(y_test_stage):
    raise ValueError(
        f'Test target row count ({len(target_rows_test)}) does not match y_test_stage length ({len(y_test_stage)}).'
    )

risk_train = target_rows_train.apply(
    lambda row: fuzzy_risk_score(row['MMSE'], row['CDRSB'], row['HIPP_TOTAL']),
    axis=1
).values
risk_test = target_rows_test.apply(
    lambda row: fuzzy_risk_score(row['MMSE'], row['CDRSB'], row['HIPP_TOTAL']),
    axis=1
).values

tf.random.set_seed(42)
np.random.seed(42)

idx = np.arange(len(X_train))
np.random.shuffle(idx)
split = int(0.8 * len(idx))
train_idx, val_idx = idx[:split], idx[split:]

X_tr, X_val = X_train[train_idx], X_train[val_idx]
y_tr, y_val = y_train_stage[train_idx], y_train_stage[val_idx]
risk_tr, risk_val = risk_train[train_idx], risk_train[val_idx]

def kde_fuzzy_priors(query_risk, ref_risk, ref_stage, bandwidth):
    query_risk = np.asarray(query_risk, dtype=float).reshape(-1)
    ref_risk = np.asarray(ref_risk, dtype=float).reshape(-1)
    ref_stage = np.asarray(ref_stage, dtype=int).reshape(-1)

    if len(ref_risk) == 0:
        return np.full((len(query_risk), 3), 1.0 / 3.0)

    priors = np.zeros((len(query_risk), 3), dtype=float)
    for i, q in enumerate(query_risk):
        d = (ref_risk - q) / max(float(bandwidth), 1e-6)
        w = np.exp(-0.5 * d * d)

        # Weighted class evidence + Laplace smoothing for stability.
        for c in range(3):
            priors[i, c] = w[ref_stage == c].sum() + 1.0

        s = priors[i].sum()
        if s <= 0:
            priors[i] = np.array([1 / 3, 1 / 3, 1 / 3], dtype=float)
        else:
            priors[i] = priors[i] / s

    return priors

# Tune fuzzy bandwidth strictly on train->validation.
bw_candidates = np.linspace(0.02, 0.30, 29)
best_bw = 0.10
best_fuzzy_val_acc = -1.0
for bw in bw_candidates:
    fuzzy_val_candidate = kde_fuzzy_priors(risk_val, risk_tr, y_tr, bw)
    fuzzy_val_pred = np.argmax(fuzzy_val_candidate, axis=1)
    acc = accuracy_score(y_val, fuzzy_val_pred)
    if acc > best_fuzzy_val_acc:
        best_fuzzy_val_acc = acc
        best_bw = float(bw)

# Fuzzy priors for fusion tuning and final test evaluation.
fuzzy_val = kde_fuzzy_priors(risk_val, risk_tr, y_tr, best_bw)
fuzzy_priors_train = kde_fuzzy_priors(risk_train, risk_train, y_train_stage, best_bw)
fuzzy_priors_test = kde_fuzzy_priors(risk_test, risk_train, y_train_stage, best_bw)
fuzzy_stage_seq_pred = np.argmax(fuzzy_priors_test, axis=1)

cls_model = models.Sequential([
    layers.Input(shape=(X_train.shape[1], X_train.shape[2])),
    layers.LSTM(64, return_sequences=True),
    layers.Dropout(0.2),
    layers.LSTM(32),
    layers.Dropout(0.2),
    layers.Dense(16, activation='relu'),
    layers.Dense(3, activation='softmax'),
])
cls_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

es = callbacks.EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True)
_ = cls_model.fit(
    X_tr, y_tr,
    validation_data=(X_val, y_val),
    epochs=60,
    batch_size=32,
    callbacks=[es],
    verbose=0
)

lstm_proba_val = cls_model.predict(X_val, verbose=0)
lstm_proba_test = cls_model.predict(X_test, verbose=0)

alpha_candidates = np.linspace(0.0, 1.0, 41)
best_alpha = 1.0
best_val_acc = -1.0
for a in alpha_candidates:
    fused_val = a * lstm_proba_val + (1.0 - a) * fuzzy_val
    fused_val_pred = np.argmax(fused_val, axis=1)
    acc = accuracy_score(y_val, fused_val_pred)
    if acc > best_val_acc:
        best_val_acc = acc
        best_alpha = float(a)

ALPHA = best_alpha
fused_proba_test = ALPHA * lstm_proba_test + (1.0 - ALPHA) * fuzzy_priors_test
fused_stage_pred = np.argmax(fused_proba_test, axis=1)

print('KDE fuzzy priors (train/test):', fuzzy_priors_train.shape, fuzzy_priors_test.shape)
print('Best fuzzy bandwidth          :', round(best_bw, 4))
print('Best fuzzy val accuracy       :', round(best_fuzzy_val_acc, 4))
print('LSTM probability shape (test) :', lstm_proba_test.shape)
print('Best alpha from validation    :', ALPHA)
print('Best fusion val accuracy      :', round(best_val_acc, 4))

display(test_df[['PTID', 'VISCODE', 'MMSE', 'CDRSB', 'HIPP_TOTAL', 'RISK_SCORE', 'FUZZY_STAGE_PRED']].head(15))

KDE fuzzy priors (train/test): (2140, 3) (496, 3)
Best fuzzy bandwidth          : 0.02
Best fuzzy val accuracy       : 0.5958
LSTM probability shape (test) : (496, 3)
Best alpha from validation    : 0.15000000000000002
Best fusion val accuracy      : 0.7921


,PTID,VISCODE,MMSE,CDRSB,HIPP_TOTAL,RISK_SCORE,FUZZY_STAGE_PRED
0,002_S_2132,sc,0.903226,0.144444,0.205473,0.862996,2
1,002_S_4473,init,0.967742,0.055556,0.168806,0.868722,2
2,002_S_6066,sc,1.000000,0.000000,0.164412,0.869365,2
3,002_S_6066,bl,0.922581,0.111111,0.179737,0.867080,2
4,002_S_6070,sc,0.967742,0.000000,0.198050,0.864204,2
5,002_S_6456,sc,1.000000,0.000000,0.184844,0.866294,2
6,002_S_6456,bl,0.980645,0.033333,0.136787,0.873167,2
7,002_S_6652,sc,0.967742,0.138889,0.158919,0.870154,2
8,002_S_6652,bl,0.980645,0.022222,0.136787,0.873167,2
9,002_S_6680,sc,1.000000,0.000000,0.253758,0.828579,2


## Where fuzzy math is used in the hybrid pipeline

### Fuzzy risk scoring (used in `fuzzy_risk_score`)
For each aligned row, the fuzzy system computes a crisp risk value:

$$r_i = \mathcal{F}(\mathrm{MMSE}_i,\mathrm{CDRSB}_i,\mathrm{HIPP}_i)$$

where $\mathcal{F}$ is the Mamdani system defined above (MFs + rules + centroid defuzzification).

### KDE fuzzy class priors (used in `kde_fuzzy_priors`)
Given query risk $q$ and reference risks $r_j$ with class labels $y_j \in \{0,1,2\}$:

$$w_j(q)=\exp\left(-\frac{1}{2}\left(\frac{r_j-q}{h}\right)^2\right)$$

$$\tilde{p}_c(q)=\sum_{j:y_j=c} w_j(q) + 1$$

$$p_c(q)=\frac{\tilde{p}_c(q)}{\sum_{k=0}^2 \tilde{p}_k(q)}$$

This produces a 3-class fuzzy prior vector per sequence.

### Hybrid fusion (used in `fused_proba_test`)
Final prediction probabilities are fused as:

$$\mathbf{p}_{\mathrm{fused}} = \alpha\,\mathbf{p}_{\mathrm{LSTM}} + (1-\alpha)\,\mathbf{p}_{\mathrm{fuzzy}}$$

with $\alpha$ selected on validation accuracy only.

In [15]:
target_names = ['CN', 'MCI', 'AD']

fuzzy_acc = accuracy_score(y_test_stage, fuzzy_stage_seq_pred)
lstm_acc = accuracy_score(y_test_stage, np.argmax(lstm_proba_test, axis=1))
fused_acc = accuracy_score(y_test_stage, fused_stage_pred)

print('Fuzzy-only sequence accuracy:', round(fuzzy_acc, 4))
print('LSTM-only sequence accuracy :', round(lstm_acc, 4))
print('Fused sequence accuracy     :', round(fused_acc, 4))

print('\nFused classification report:')
print(classification_report(y_test_stage, fused_stage_pred, target_names=target_names, zero_division=0))

cm_fused = confusion_matrix(y_test_stage, fused_stage_pred, labels=[0, 1, 2])
cm_df = pd.DataFrame(cm_fused, index=[f'True_{c}' for c in target_names], columns=[f'Pred_{c}' for c in target_names])
display(cm_df)

metrics_summary = {
    'fuzzy_accuracy': float(fuzzy_acc),
    'lstm_accuracy': float(lstm_acc),
    'fused_accuracy': float(fused_acc),
    'alpha_validation_best': float(ALPHA),
    'fuzzy_kde_bandwidth': float(best_bw),
    'fuzzy_val_accuracy_best': float(best_fuzzy_val_acc),
}
print('Metrics summary:', metrics_summary)

Fuzzy-only sequence accuracy: 0.6351
LSTM-only sequence accuracy : 0.8085
Fused sequence accuracy     : 0.8125

Fused classification report:
              precision    recall  f1-score   support

          CN       0.90      0.72      0.80       137
         MCI       0.74      0.81      0.78       200
          AD       0.84      0.89      0.87       159

    accuracy                           0.81       496
   macro avg       0.83      0.81      0.81       496
weighted avg       0.82      0.81      0.81       496



,Pred_CN,Pred_MCI,Pred_AD
True_CN,99,38,0
True_MCI,11,163,26
True_AD,0,18,141


Metrics summary: {'fuzzy_accuracy': 0.6350806451612904, 'lstm_accuracy': 0.8084677419354839, 'fused_accuracy': 0.8125, 'alpha_validation_best': 0.15000000000000002, 'fuzzy_kde_bandwidth': 0.02, 'fuzzy_val_accuracy_best': 0.5957943925233645}


## Hybrid Fusion (Implemented)

This notebook now runs full hybrid fusion in code:

- Trains an LSTM classifier on sequence tensors
- Builds fuzzy risk scores and calibrated fuzzy priors from train risk bins
- Tunes `alpha` on validation data
- Reports raw fusion results exactly as computed (no forced fallback)

Fusion formula used:

- `P_fused = alpha * P_lstm + (1 - alpha) * P_fuzzy`

In [16]:
OUT_DIR = Path('..') / 'outputs'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Save visit-level fuzzy outputs
test_df.to_csv(OUT_DIR / 'test_with_fuzzy_scores.csv', index=False)

# Save sequence-level fusion outputs
fusion_df = pd.DataFrame({
    'y_true_stage': y_test_stage.astype(int),
    'fuzzy_stage_pred': fuzzy_stage_seq_pred.astype(int),
    'lstm_stage_pred': np.argmax(lstm_proba_test, axis=1).astype(int),
    'fused_stage_pred': fused_stage_pred.astype(int),
    'lstm_prob_cn': lstm_proba_test[:, 0],
    'lstm_prob_mci': lstm_proba_test[:, 1],
    'lstm_prob_ad': lstm_proba_test[:, 2],
    'fuzzy_prior_cn': fuzzy_priors_test[:, 0],
    'fuzzy_prior_mci': fuzzy_priors_test[:, 1],
    'fuzzy_prior_ad': fuzzy_priors_test[:, 2],
    'fused_prob_cn': fused_proba_test[:, 0],
    'fused_prob_mci': fused_proba_test[:, 1],
    'fused_prob_ad': fused_proba_test[:, 2],
})
fusion_df.to_csv(OUT_DIR / 'hybrid_fusion_predictions.csv', index=False)

with open(OUT_DIR / 'hybrid_metrics.json', 'w', encoding='utf-8') as f:
    json.dump(metrics_summary, f, indent=2)

print('Saved:', (OUT_DIR / 'test_with_fuzzy_scores.csv').resolve())
print('Saved:', (OUT_DIR / 'hybrid_fusion_predictions.csv').resolve())
print('Saved:', (OUT_DIR / 'hybrid_metrics.json').resolve())

Saved: C:\repos\adni-project\outputs\test_with_fuzzy_scores.csv
Saved: C:\repos\adni-project\outputs\hybrid_fusion_predictions.csv
Saved: C:\repos\adni-project\outputs\hybrid_metrics.json
